## LAB ASSIGNMENT

Name: Joseph Atem Deng Aruei

Nim: 244107020242

**1. Differentiate usable and unusable variables**

In [1]:
import pandas as pd

df = pd.read_csv("wbc.csv")

print(df.info())
print(df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

- Usable input variables: the 30 numerical medical-measurement columns, from radius_mean to fractal_dimension_worst.
- Unusable variable: id, because it is only a patient/record identifier.
- Unusable variable: Unnamed: 32, because it is empty for every row.
- Target variable: diagnosis, because it is the outcome we want to predict.

In [2]:
df = df.drop(columns=["id", "Unnamed: 32"])

**2. Encode the diagnosis column**

In [3]:
df["diagnosis"] = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

print(df["diagnosis"].value_counts())

diagnosis
0    357
1    212
Name: count, dtype: int64


## Answers:
- B (Benign) becomes 0
- M (Malignant) becomes 1

**3. Separate inputs and target, then standardize numerical variables**

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns="diagnosis")
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Answer:
All 30 usable input columns are numerical, so all 30 are standardized. The scaler is fitted only on training data to prevent data leakage.

**4. Perform feature extraction**

In [5]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()].tolist()

print("Selected features:")
for feature in selected_features:
    print("-", feature)

Selected features:
- radius_mean
- perimeter_mean
- area_mean
- concavity_mean
- concave points_mean
- radius_worst
- perimeter_worst
- area_worst
- concavity_worst
- concave points_worst


**5. Train and test Logistic Regression**

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=10000, random_state=42)

model.fit(X_train_selected, y_train)

pred = model.predict(X_test_selected)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(
    y_test,
    pred,
    target_names=["Benign (B)", "Malignant (M)"]
))

Accuracy: 0.956140350877193
               precision    recall  f1-score   support

   Benign (B)       0.95      0.99      0.97        72
Malignant (M)       0.97      0.90      0.94        42

     accuracy                           0.96       114
    macro avg       0.96      0.95      0.95       114
 weighted avg       0.96      0.96      0.96       114



**6. Equivalent pipeline version**

In [7]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=10)),
    ("model", LogisticRegression(max_iter=10000, random_state=42))
])

pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.956140350877193


**7. How many optimal features and which ones?**

In [8]:
selected_features = X.columns[
    pipe.named_steps["selector"].get_support()
].tolist()

print(selected_features)

['radius_mean', 'perimeter_mean', 'area_mean', 'concavity_mean', 'concave points_mean', 'radius_worst', 'perimeter_worst', 'area_worst', 'concavity_worst', 'concave points_worst']


The model used 10 optimal features selected with SelectKBest using the ANOVA F-test. These features were the most statistically associated with cancer diagnosis.

- concave points_worst
- perimeter_worst
- radius_worst
- concave points_mean
- perimeter_mean
- area_worst
- radius_mean
- area_mean
- concavity_mean
- concavity_worst